In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable the T4 GPU runtime before continuing.")

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/contextfit_drive")

PROJECT = Path("/content/contextfit_drive/MyDrive/ContextFit")
PROJECT.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT)

Mounted at /content/contextfit_drive
Project folder: /content/contextfit_drive/MyDrive/ContextFit


In [4]:
from pathlib import Path
import hashlib
import json

UPLOAD = Path("/content")
PROJECT = Path("/content/contextfit_drive/MyDrive/ContextFit")

if not PROJECT.parent.is_dir():
    raise RuntimeError("Mount Google Drive before running this cell.")

SOURCE = PROJECT / "evaluation_src" / "contextfit"
EXPORT = PROJECT / "requests" / "export_20261008T195330_493252Z"

names = [
    "inference.py",
    "benchmark.py",
    "validation.py",
    "dev_requests.jsonl",
    "dev_diagnostics.jsonl",
    "export_manifest.json",
]

contents = {}
for name in names:
    path = UPLOAD / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing: {path}")
    contents[name] = path.read_bytes()

manifest = json.loads(contents["export_manifest.json"])

for name in ("dev_requests.jsonl", "dev_diagnostics.jsonl"):
    actual = hashlib.sha256(contents[name]).hexdigest()
    if actual != manifest["output_sha256"][name]:
        raise ValueError(f"Hash mismatch: {name}")

actual = hashlib.sha256(contents["validation.py"]).hexdigest()
expected = manifest["source_sha256"]["src/contextfit/validation.py"]
if actual != expected:
    raise ValueError("validation.py differs from the export version.")

SOURCE.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
(SOURCE / "__init__.py").touch()

for name, content in contents.items():
    destination = SOURCE / name if name.endswith(".py") else EXPORT / name
    destination.write_bytes(content)
    print("Saved:", destination)

print("\nPASS: files saved to Drive and export hashes verified.")

Saved: /content/contextfit_drive/MyDrive/ContextFit/evaluation_src/contextfit/inference.py
Saved: /content/contextfit_drive/MyDrive/ContextFit/evaluation_src/contextfit/benchmark.py
Saved: /content/contextfit_drive/MyDrive/ContextFit/evaluation_src/contextfit/validation.py
Saved: /content/contextfit_drive/MyDrive/ContextFit/requests/export_20261008T195330_493252Z/dev_requests.jsonl
Saved: /content/contextfit_drive/MyDrive/ContextFit/requests/export_20261008T195330_493252Z/dev_diagnostics.jsonl
Saved: /content/contextfit_drive/MyDrive/ContextFit/requests/export_20261008T195330_493252Z/export_manifest.json

PASS: files saved to Drive and export hashes verified.


## load the model and check all 18 development prompts

In [5]:
import sys
import json
import torch
import transformers
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT = Path("/content/contextfit_drive/MyDrive/ContextFit")
SOURCE_ROOT = PROJECT / "evaluation_src"
EXPORT = PROJECT / "requests/export_20261008T195330_493252Z"

if not torch.cuda.is_available():
    raise RuntimeError("Enable the T4 GPU runtime first.")

sys.path.insert(0, str(SOURCE_ROOT))

from contextfit import inference, benchmark

if Path(inference.__file__).resolve() != (
    SOURCE_ROOT / "contextfit/inference.py"
).resolve():
    raise RuntimeError("Another contextfit module is loaded. Restart the session.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Avoid loading a second copy if you rerun this cell.
if inference.model is None:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
    ).to("cuda")
    model.eval()
    inference.configure(model, tokenizer)
else:
    model = inference.model
    tokenizer = inference.tokenizer

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Inference module:", inference.__file__)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU: Tesla T4
PyTorch: 2.11.0+cu130
Transformers: 5.18.0
Model device: cuda:0
Model dtype: torch.float16
Inference module: /content/contextfit_drive/MyDrive/ContextFit/evaluation_src/contextfit/inference.py


In [6]:
def read_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

dev_records = read_jsonl(EXPORT / "dev_requests.jsonl")
diagnostics = read_jsonl(EXPORT / "dev_diagnostics.jsonl")

expected = {
    (row["question_id"], row["policy"]): row["prompt_tokens"]
    for row in diagnostics
}

request_keys = {
    (row["question_id"], row["policy"])
    for row in dev_records
}

if (
    len(dev_records) != 18
    or len(diagnostics) != 18
    or len(request_keys) != 18
    or request_keys != set(expected)
):
    raise ValueError("Expected 18 matching, unique development requests.")

for record in dev_records:
    key = (record["question_id"], record["policy"])
    prepared = inference.prepare_inputs(
        record["question"], record["context"]
    )
    actual = prepared["input_ids"].shape[1]
    del prepared

    if actual != expected[key]:
        raise ValueError(
            f"{key}: laptop counted {expected[key]}, Colab counted {actual}"
        )

print("PASS: all 18 prompt-token counts match the Windows export.")
print("PASS: all development prompts fit the input limit.")

PASS: all 18 prompt-token counts match the Windows export.
PASS: all development prompts fit the input limit.


In [7]:
import hashlib
import platform
from datetime import datetime, timezone

# Keep the same directory when rerunning this cell in this session.
if "DEV_RUN_DIR" not in globals():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    DEV_RUN_DIR = PROJECT / "results" / f"dev_{run_id}"

DEV_RUN_DIR.mkdir(parents=True, exist_ok=True)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

config = {
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "gpu": torch.cuda.get_device_name(0),
    "dtype": str(next(model.parameters()).dtype),
    "attention": getattr(model.config, "_attn_implementation", None),
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "transformers": transformers.__version__,
    "cuda": torch.version.cuda,
    "max_new_tokens": 80,
    "do_sample": False,
    "requests_sha256": file_hash(EXPORT / "dev_requests.jsonl"),
    "source_sha256": {
        name: file_hash(SOURCE_ROOT / "contextfit" / name)
        for name in ("inference.py", "benchmark.py", "validation.py")
    },
}

(DEV_RUN_DIR / "config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

# Save the effective model generation defaults for reproducibility.
model.generation_config.save_pretrained(DEV_RUN_DIR)

print("Warming up...")
for policy in ("top_k_8", "top_k_2", "budget_900"):
    record = next(r for r in dev_records if r["policy"] == policy)
    inference.run_record(record)

print("Warm-up complete. Starting measured runs.")

counts = benchmark.run_batch(
    records=dev_records,
    predict=inference.run_record,
    output_path=DEV_RUN_DIR / "requests.jsonl",
    config=config,
    repeats=3,
)

print("\nRun counts:", counts)
print("Results folder:", DEV_RUN_DIR)

Warming up...
Warm-up complete. Starting measured runs.
OK: dev_002 | repeat 3
OK: dev_002 | repeat 2
OK: dev_003 | repeat 3
OK: dev_005 | repeat 2
OK: dev_001 | repeat 2
OK: dev_005 | repeat 3
OK: dev_006 | repeat 1
OK: dev_005 | repeat 2
OK: dev_004 | repeat 2
OK: dev_004 | repeat 1
OK: dev_004 | repeat 1
OK: dev_002 | repeat 2
OK: dev_004 | repeat 2
OK: dev_005 | repeat 3
OK: dev_006 | repeat 2
OK: dev_002 | repeat 1
OK: dev_001 | repeat 2
OK: dev_004 | repeat 3
OK: dev_002 | repeat 1
OK: dev_006 | repeat 3
OK: dev_004 | repeat 1
OK: dev_001 | repeat 3
OK: dev_002 | repeat 2
OK: dev_002 | repeat 3
OK: dev_003 | repeat 2
OK: dev_001 | repeat 1
OK: dev_003 | repeat 2
OK: dev_004 | repeat 3
OK: dev_001 | repeat 2
OK: dev_006 | repeat 2
OK: dev_006 | repeat 3
OK: dev_003 | repeat 2
OK: dev_005 | repeat 1
OK: dev_001 | repeat 3
OK: dev_005 | repeat 2
OK: dev_005 | repeat 3
OK: dev_005 | repeat 1
OK: dev_003 | repeat 3
OK: dev_006 | repeat 3
OK: dev_001 | repeat 1
OK: dev_004 | repeat 2
O

In [8]:
import pandas as pd

entries = read_jsonl(DEV_RUN_DIR / "requests.jsonl")

expected_jobs = {
    benchmark.job_id(record, repeat, config)
    for record in dev_records
    for repeat in range(1, 4)
}

successful = {
    entry["job_id"]: entry
    for entry in entries
    if entry["status"] == "ok"
    and entry["job_id"] in expected_jobs
}

missing = expected_jobs - successful.keys()
print(f"Successful jobs: {len(successful)}/54")

if missing:
    errors = [
        entry for entry in entries
        if entry["status"] == "error" and entry["job_id"] in missing
    ]
    for entry in errors[-5:]:
        print(entry["error_type"], entry["error"])
    raise RuntimeError("Some jobs are incomplete. Share the errors above.")

rows = [
    {**entry["result"], "repeat": entry["repeat"]}
    for entry in successful.values()
]
df = pd.DataFrame(rows)

summary = df.groupby("policy").agg(
    runs=("generation_seconds", "size"),
    median_input_tokens=("input_tokens", "median"),
    median_output_tokens=("output_tokens", "median"),
    median_seconds=("generation_seconds", "median"),
    max_peak_allocated_gib=("peak_allocated_gib", "max"),
)

display(summary.round(3))

print("\nDistinct answers for each question and policy:")
for (qid, policy), group in df.groupby(["question_id", "policy"]):
    print(f"\n{qid} | {policy}")
    for answer in group["answer"].unique():
        print(answer)

print("\nGenerations reaching the 80-token cap:",
      int((df["output_tokens"] >= 80).sum()))

Successful jobs: 54/54


,runs,median_input_tokens,median_output_tokens,median_seconds,max_peak_allocated_gib
policy,,,,,
budget_900,18,902.0,33.5,1.413,3.044
top_k_2,18,630.0,45.0,2.020,2.968
top_k_8,18,2290.5,29.5,2.144,3.640



Distinct answers for each question and policy:

dev_001 | budget_900
Applications may require specific versions of libraries due to bugs being fixed or outdated interfaces. This makes it impossible for one Python installation to satisfy all applications' requirements simultaneously. Using separate virtual environments allows each application to have its own isolated Python installation and dependencies.

dev_001 | top_k_2
Separate virtual environments are used for applications to ensure compatibility between them when they require different versions of libraries. This prevents conflicts where one application might need a specific version of a package that another application cannot support due to differing requirements. By creating distinct virtual environments for each application, developers can independently manage their dependencies without affecting others' installations.

dev_001 | top_k_8
Use separate virtual environments for applications because different applications may requ

In [9]:
for qid in ("dev_002", "dev_003", "dev_005"):
    print("\n" + "=" * 80)
    print("QUESTION:", qid)

    for record in dev_records:
        if record["question_id"] != qid:
            continue

        print("\nPOLICY:", record["policy"])
        print("QUESTION TEXT:", record["question"])
        print("SOURCES:", record["source_ids"])
        print("CONTEXT:\n", record["context"])
        print("-" * 80)


QUESTION: dev_002

POLICY: top_k_8
QUESTION TEXT: What does the PYTHONPATH variable do?
SOURCES: ['python_venv_chunk003', 'python_modules_chunk010', 'python_modules_chunk007', 'python_modules_chunk009', 'python_errors_chunk005', 'python_errors_chunk007', 'python_modules_chunk005', 'python_errors_chunk011']
CONTEXT:
 [python_venv_chunk003] Activating the virtual environment will change your shell’s prompt to show what
virtual environment you’re using, and modify the environment so that running python will get you that particular version and installation of Python.
For example:

$ source ~/envs/tutorial-env/bin/activate
(tutorial-env) $ python
Python 3.5.1 (default, May  6 2016, 10:59:36)
  ...
>>> import sys
>>> sys.path
['', '/usr/local/lib/python35.zip', ...,
'~/envs/tutorial-env/lib/python3.5/site-packages']
>>>

Note that the activated virtual environment does not alter the PYTHONPATH variable in any way.
This may lead to unexpected results if the path includes references to code w

In [10]:
from google.colab import files

files.download(str(EXPORT / "dev_requests.jsonl"))
files.download(str(DEV_RUN_DIR / "requests.jsonl"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files
import hashlib
import json
import re

uploaded = files.upload()

test_uploads = {}
for name, content in uploaded.items():
    clean_name = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", name)
    if clean_name in test_uploads:
        raise ValueError(f"Duplicate upload: {clean_name}")
    test_uploads[clean_name] = content

required = ("test_requests.jsonl", "test_diagnostics.jsonl")
manifest = json.loads(
    (EXPORT / "export_manifest.json").read_text(encoding="utf-8")
)

for name in required:
    if name not in test_uploads:
        raise ValueError(f"Missing upload: {name}")

    actual_hash = hashlib.sha256(test_uploads[name]).hexdigest()
    if actual_hash != manifest["output_sha256"][name]:
        raise ValueError(f"{name} does not match the original export.")

test_records = [
    json.loads(line)
    for line in test_uploads["test_requests.jsonl"].decode("utf-8").splitlines()
    if line.strip()
]
test_diagnostics = [
    json.loads(line)
    for line in test_uploads["test_diagnostics.jsonl"].decode("utf-8").splitlines()
    if line.strip()
]

expected = {
    (row["question_id"], row["policy"]): row["prompt_tokens"]
    for row in test_diagnostics
}
keys = {(r["question_id"], r["policy"]) for r in test_records}

if (
    len(test_records) != 72
    or len(test_diagnostics) != 72
    or len(keys) != 72
    or keys != set(expected)
):
    raise ValueError("Expected 72 unique, matching test requests.")

for record in test_records:
    key = (record["question_id"], record["policy"])
    prepared = inference.prepare_inputs(
        record["question"], record["context"]
    )
    actual_tokens = prepared["input_ids"].shape[1]
    del prepared

    if actual_tokens != expected[key]:
        raise ValueError(
            f"{key}: Windows={expected[key]}, Colab={actual_tokens}"
        )

for name in required:
    (EXPORT / name).write_bytes(test_uploads[name])

print("PASS: both test files match the original export.")
print("PASS: all 72 test prompt-token counts match.")
print("PASS: test files saved to Drive.")

Saving test_diagnostics.jsonl to test_diagnostics.jsonl
Saving test_requests.jsonl to test_requests.jsonl
PASS: both test files match the original export.
PASS: all 72 test prompt-token counts match.
PASS: test files saved to Drive.


In [12]:
from datetime import datetime, timezone
import json

# Preserve the directory when rerunning this cell in the same session.
if "TEST_RUN_DIR" not in globals():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    TEST_RUN_DIR = PROJECT / "results" / f"test_{run_id}"

TEST_RUN_DIR.mkdir(parents=True, exist_ok=True)

test_config = {
    **config,
    "split": "test",
    "requests_sha256": file_hash(EXPORT / "test_requests.jsonl"),
}

# Ensure the inference code has not changed since development.
for name, expected_hash in config["source_sha256"].items():
    actual_hash = file_hash(SOURCE_ROOT / "contextfit" / name)
    if actual_hash != expected_hash:
        raise RuntimeError(f"{name} changed after development.")

(TEST_RUN_DIR / "config.json").write_text(
    json.dumps(test_config, indent=2),
    encoding="utf-8",
)
model.generation_config.save_pretrained(TEST_RUN_DIR)

print("Results folder:", TEST_RUN_DIR)
print("Warming up; these generations are excluded from results.")

for policy in ("top_k_8", "top_k_2", "budget_900"):
    record = next(r for r in dev_records if r["policy"] == policy)
    inference.run_record(record)

counts = benchmark.run_batch(
    records=test_records,
    predict=inference.run_record,
    output_path=TEST_RUN_DIR / "requests.jsonl",
    config=test_config,
    repeats=3,
)

print("\nRun counts:", counts)
print("Saved to:", TEST_RUN_DIR / "requests.jsonl")

Results folder: /content/contextfit_drive/MyDrive/ContextFit/results/test_20261009T062709_302329Z
Warming up; these generations are excluded from results.
OK: test_013 | repeat 1
OK: test_001 | repeat 3
OK: test_011 | repeat 1
OK: test_003 | repeat 2
OK: test_021 | repeat 2
OK: test_016 | repeat 1
OK: test_001 | repeat 2
OK: test_019 | repeat 2
OK: test_018 | repeat 1
OK: test_011 | repeat 3
OK: test_005 | repeat 3
OK: test_012 | repeat 3
OK: test_006 | repeat 2
OK: test_022 | repeat 3
OK: test_023 | repeat 1
OK: test_005 | repeat 1
OK: test_005 | repeat 2
OK: test_002 | repeat 1
OK: test_020 | repeat 3
OK: test_009 | repeat 3
OK: test_008 | repeat 1
OK: test_018 | repeat 2
OK: test_021 | repeat 3
OK: test_010 | repeat 3
OK: test_015 | repeat 3
OK: test_015 | repeat 1
OK: test_015 | repeat 2
OK: test_001 | repeat 1
OK: test_012 | repeat 2
OK: test_004 | repeat 3
OK: test_022 | repeat 3
OK: test_002 | repeat 1
OK: test_006 | repeat 1
OK: test_013 | repeat 2
OK: test_015 | repeat 1
OK: t

In [13]:
import pandas as pd

entries = read_jsonl(TEST_RUN_DIR / "requests.jsonl")

expected_jobs = {
    benchmark.job_id(record, repeat, test_config)
    for record in test_records
    for repeat in range(1, 4)
}

successful = {
    entry["job_id"]: entry
    for entry in entries
    if entry["status"] == "ok"
    and entry["job_id"] in expected_jobs
}

missing = expected_jobs - successful.keys()
print(f"Successful jobs: {len(successful)}/216")

if missing:
    errors = [
        entry for entry in entries
        if entry["status"] == "error"
        and entry["job_id"] in missing
    ]
    for entry in errors[-5:]:
        print(entry["error_type"], entry["error"])
    raise RuntimeError("Some test jobs are incomplete. Share the errors.")

test_df = pd.DataFrame([
    {**entry["result"], "repeat": entry["repeat"]}
    for entry in successful.values()
])

summary = test_df.groupby("policy").agg(
    runs=("generation_seconds", "size"),
    median_input_tokens=("input_tokens", "median"),
    median_output_tokens=("output_tokens", "median"),
    median_seconds=("generation_seconds", "median"),
    max_peak_allocated_gib=("peak_allocated_gib", "max"),
    capped_generations=("output_tokens", lambda values: (values >= 80).sum()),
)

display(summary.round(3))

answer_variants = test_df.groupby(
    ["question_id", "policy"]
)["answer"].nunique()

print(
    "Question-policy pairs with different answers across repeats:",
    int((answer_variants > 1).sum()),
)
print("Results:", TEST_RUN_DIR / "requests.jsonl")

Successful jobs: 216/216


,runs,median_input_tokens,median_output_tokens,median_seconds,max_peak_allocated_gib,capped_generations
policy,,,,,,
budget_900,72,910.5,51.0,2.147,3.044,9
top_k_2,72,627.5,43.5,1.691,2.974,6
top_k_8,72,2235.0,36.0,2.335,3.651,9


Question-policy pairs with different answers across repeats: 0
Results: /content/contextfit_drive/MyDrive/ContextFit/results/test_20261009T062709_302329Z/requests.jsonl


In [14]:
from google.colab import files
import shutil

download_path = "/content/contextfit_test_results.jsonl"
shutil.copyfile(TEST_RUN_DIR / "requests.jsonl", download_path)
files.download(download_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>